# Макроэкономические показатели Росстата

Самостоятельный блокнот: установка зависимостей → загрузка Excel → разбор таблиц → проверка → CSV.
Все ссылки и код находятся здесь. Дополнительно нужен только файл `certificates/rosstat_ca_bundle.pem`.
Запустите **Run All** в Python 3.12+; первый запуск скачает файлы, последующие используют кеш.

Срез источников — октябрь 2026 года. Получаем **11 месячных и 3 квартальных ряда**:

| Показатели | Начало истории |
|---|---|
| Общий ИПЦ, продовольствие, непродовольственные товары, услуги | 1991 |
| Цены производителей: промышленность, добыча, обработка (ОКВЭД2) | 2013 |
| Номинальная зарплата по экономике, рублей в месяц | 1998 |
| Номинальный оборот розницы, млн рублей | 2026 |
| Физический объём платных услуг: месяц к месяцу и год к году | 2022 |
| ВВП в ценах 2021 года, с сезонной корректировкой и без; индекс год к году | 2011/2012 |

Индексы с базой 100 сохраняются как индексы. Кварталы хранятся отдельно от месяцев.
Безработица, промпроизводство, базовый ИПЦ, доходы и инвестиции пока не включены.

## 1. Зависимости
Устанавливаются в среду текущего ядра. Jupyter должен быть уже запущен.

In [1]:
%pip install --quiet --disable-pip-version-check requests==2.34.2 pandas==3.1.0 openpyxl==3.1.5 xlrd==2.0.2

Note: you may need to restart the kernel to use updated packages.


## 2. Настройки и источники
`ALLOW_NETWORK=False` запрещает новые загрузки; `REFRESH=True` обновляет снимки.
Исходные Excel сохраняются с понятными именами в `data/raw/`, результаты — в новой папке `data/processed/`.
Файлы на сайте зафиксированы явно: при появлении нового выпуска ссылки и структуру нужно проверить.

In [2]:
from pathlib import Path
from datetime import datetime, timezone
from urllib.parse import urljoin, urlparse
import hashlib
import json
import math
import re
import time
import warnings

import pandas as pd
import requests
from IPython.display import display

ROOT = Path.cwd()
if not (ROOT / 'certificates/rosstat_ca_bundle.pem').is_file():
    raise RuntimeError('Откройте блокнот из папки проекта рядом с certificates/')
RAW = ROOT / 'data/raw'
RAW.mkdir(parents=True, exist_ok=True)
OUT = ROOT / 'data/processed' / datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
ALLOW_NETWORK = True
REFRESH = False
BASE_URL = 'https://rosstat.gov.ru/storage/mediabank/'
MONTHS = 'январь февраль март апрель май июнь июль август сентябрь октябрь ноябрь декабрь'.split()

FILES = {
    'cpi': 'ipc_mes_09-2026.xlsx',
    'cpi_check': 'ipc_RF_fo_sub_09-2026.xlsx',
    'ppi': 'Proizvoditeli_Ind_VED_09-2026.xlsx',
    'wages': 'tab1-zpl_08-2026.xlsx',
    'retail': 'Oborot_m_09-2026.xlsx',
    'gdp': 'VVP_kvartal_s-1995-2026.xlsx',
}
SERVICE_FILES = {
    2022: [
        'plat_012022.xls', 'plat_022022.xls', 'plat_032022.xls',
        'plat_042022.xls', 'plat_052022.xls', 'plat_062022.xls',
        'plat_072022.xls', 'plat_082022.xls', 'plat_092022.xls',
        'plat_102022.xls', 'plat_112022.xls', 'Plat_122022.xls',
    ],
    2023: [
        'Plat_012023.xls', 'Plat_022023.xls', 'Plat_032023.xls',
        'Plat_042023.xls', 'Plat_052023.xls', 'Plat_062023.xls',
        'Plat_072023.xls', 'Plat_082023.xls', 'Plat_092023.xls',
        'Plat_102023.xls', 'Plat_112023.xls', 'Plat_122023.xls',
    ],
    2024: [
        'plat_012024.xls', 'plat-022024.xls', 'plat_032024.xls',
        'plat_042024.xls', 'plat-052024.xls', 'Plat_062024.xls',
        'Plat_072024.xls', 'Plat_082024.xls', 'plat_092024.xls',
        'plat_102024.xls', 'plat_112024.xls', 'Plat_122024.xls',
    ],
    2025: [
        'plat_01_2025.xls', 'plat_02_2025.xls', 'plat_03_2025.xls',
        'plat_04_2025.xls', 'plat_05_2025.xls', 'plat_06_2025.xls',
        'plat_07_2025.xls', 'plat_08_2025.xls', 'plat_09_2025.xls',
        'plat_10_2025.xls', 'plat_11_2025.xls', 'plat_12_2025.xls',
    ],
    2026: [
        'plat_01_2026.xls', 'plat_02_2026.xls', 'plat_03_2026.xls',
        'plat_04_2026.xls', 'plat_05_2026.xls', 'plat_06_2026.xls',
        'plat_07_2026.xls', 'plat_08_2026.xls', 'plat_092026.xls',
    ],
}
SERVICES = [(year, month, name) for year, names in SERVICE_FILES.items()
            for month, name in enumerate(names, 1)]
FILE_NAMES = list(FILES.values()) + [name for _, _, name in SERVICES]
print(f'Источников: {len(FILE_NAMES)}; месячных выпусков услуг: {len(SERVICES)}')

Источников: 63; месячных выпусков услуг: 57


## 3. Загрузка Excel
TLS проверяется с официальной цепочкой Минцифры из папки `certificates/`.
Для каждого файла сохраняются URL, SHA-256 и время получения. При чтении кеша проверяется хеш.
При ошибке загрузки или изменении формата выполнение останавливается.

In [3]:
http = requests.Session()
http.verify = str(ROOT / 'certificates/rosstat_ca_bundle.pem')
http.headers['User-Agent'] = 'rosstat-macro-research/1.0'
manifest_path = RAW / 'downloads.json'
metadata = json.loads(manifest_path.read_text()) if manifest_path.exists() else {}


def download(name):
    path, url = RAW / name, BASE_URL + name
    if path.exists() and not REFRESH:
        digest = hashlib.sha256(path.read_bytes()).hexdigest()
        if name in metadata and metadata[name]['sha256'] != digest:
            raise ValueError(f'Изменился кеш {name}: загрузите заново с REFRESH=True')
        metadata.setdefault(name, dict(url=url, sha256=digest, retrieved_at=None,
                                       acquisition='local_file'))
        return path
    if not ALLOW_NETWORK:
        raise FileNotFoundError(f'Нет {name}; разрешите сеть или положите файл в {RAW}')
    response = http.get(url, timeout=(15, 90), allow_redirects=False)
    for _ in range(5):
        if response.status_code not in (301, 302, 303, 307, 308):
            break
        url = urljoin(url, response.headers['Location'])
        if urlparse(url).scheme != 'https' or urlparse(url).hostname not in {'rosstat.gov.ru', 'www.rosstat.gov.ru'}:
            raise ValueError(f'Неожиданное перенаправление: {url}')
        response = http.get(url, timeout=(15, 90), allow_redirects=False)
    response.raise_for_status()
    payload = response.content
    signature = b'PK\x03\x04' if name.endswith('.xlsx') else bytes.fromhex('D0CF11E0A1B11AE1')
    if response.status_code != 200 or not payload.startswith(signature):
        raise ValueError(f'{name}: вместо Excel получен другой ответ')
    temporary = path.with_suffix(path.suffix + '.part')
    temporary.write_bytes(payload)
    temporary.replace(path)
    metadata[name] = dict(url=BASE_URL + name, sha256=hashlib.sha256(payload).hexdigest(),
                          retrieved_at=datetime.now(timezone.utc).isoformat(),
                          acquisition='https_download')
    manifest_path.write_text(json.dumps(metadata, ensure_ascii=False, indent=2))
    time.sleep(0.3)
    return path


paths = {name: download(name) for name in FILE_NAMES}
manifest_path.write_text(json.dumps(metadata, ensure_ascii=False, indent=2))
print(f'Готово: {len(paths)} файлов. Кеш: {RAW}')

Готово: 63 файлов. Кеш: /Users/oleynik_n_a/Documents/ChatGPT/Годовой проект/data/raw


## 4. Общие функции для чтения таблиц
Пропуски сохраняются, неизвестные числовые пометки вызывают ошибку.
Контрольные заголовки защищают от незаметной смены структуры таблиц.

In [4]:
def text(value):
    if pd.isna(value):
        return ''
    return re.sub(r'\s+', ' ', str(value).replace('\xa0', ' ').replace('\u202f', ' ')).strip()


def number(value):
    label = text(value).lower().replace(' ', '').replace(',', '.').replace('−', '-')
    if label in {'', '-', '–', '—', '...', '…', 'н/д', 'н.д.', 'nan', 'none'}:
        return float('nan')
    if not re.fullmatch(r'[+-]?\d+(?:\.\d+)?', label):
        raise ValueError(f'Нераспознанное число: {value!r}')
    return float(label)


def year_number(value):
    label = text(value)
    if re.fullmatch(r'(?:19|20)\d{2}(?:\.0|\s*г\.?)?(?:\s*(?:\(\d+\)|\d+\)|\*+))?', label):
        return int(label[:4])
    return None


def expect(frame, row, column, label, contains=False):
    actual = text(frame.iat[row, column])
    if not (label.casefold() in actual.casefold() if contains else actual.casefold() == label.casefold()):
        raise ValueError(f'Изменился заголовок [{row}, {column}]: {actual!r}; ожидалось {label!r}')


source_notes = set()
parts = []


def read_excel(name, sheet):
    with warnings.catch_warnings():
        warnings.filterwarnings('ignore', message='The xlrd engine is deprecated.*')
        frame = pd.read_excel(paths[name], sheet_name=sheet, header=None, dtype=object)
    for value in frame.to_numpy().ravel():
        label = text(value)
        if len(label) > 70 and re.search('Без учета|учетом|деноминац|пересматрив|Данные содержат', label, re.I):
            source_notes.add((name, sheet, label))
    return frame


def add_series(records, indicator, frequency, unit, basis, name, sheet, note=''):
    result = pd.DataFrame(records, columns=['period', 'value']).sort_values('period')
    observed = result.loc[result.value.notna(), 'period']
    if observed.empty or result.period.duplicated().any():
        raise ValueError(f'{indicator}: нет значений или повторяются периоды')
    if not all(math.isfinite(v) for v in result.value.dropna()):
        raise ValueError(f'{indicator}: бесконечные значения')
    result = result.loc[result.period.between(observed.min(), observed.max())].copy()
    for key, value in dict(indicator=indicator, frequency=frequency, unit=unit, basis=basis,
                           territory='Российская Федерация', source_url=BASE_URL + name,
                           source_sha256=metadata[name]['sha256'], sheet=sheet,
                           retrieved_at=metadata[name]['retrieved_at'],
                           acquisition=metadata[name]['acquisition'], methodology_note=note).items():
        result[key] = value
    result['release_date'] = pd.NaT
    parts.append(result)

## 5. ИПЦ: общий индекс и три группы
На листах `01`–`04` годы расположены по столбцам, отдельные месяцы — по строкам.
Берём только блок «к концу предыдущего месяца», исключая итог к декабрю прошлого года.

In [5]:
CPI_INDICATORS = ['cpi_mom_index', 'cpi_food_mom_index',
                  'cpi_nonfood_mom_index', 'cpi_services_mom_index']
name = FILES['cpi']
for sheet, indicator in zip(['01', '02', '03', '04'], CPI_INDICATORS):
    frame = read_excel(name, sheet)
    expect(frame, 0, 0, 'Индексы потребительских цен', contains=True)
    expect(frame, 0, 0, 'Российской Федерации', contains=True)
    expect(frame, 4, 0, 'к концу предыдущего месяца')
    records = []
    for column, year in enumerate(range(1991, 2027), 1):
        if year_number(frame.iat[3, column]) != year:
            raise ValueError(f'Изменился год ИПЦ: столбец {column}')
        for month in range(1, 13):
            row = month + 4
            expect(frame, row, 0, MONTHS[month - 1])
            records.append((pd.Timestamp(year=year, month=month, day=1), number(frame.iat[row, column])))
    add_series(records, indicator, 'monthly', '%', 'предыдущий месяц = 100', name, sheet,
               'Индекс на конец месяца; территориальные оговорки сохранены в source_notes.csv.')
print('ИПЦ: извлечены 4 ряда.')

ИПЦ: извлечены 4 ряда.


## 6. Цены производителей
Используем единую классификацию ОКВЭД2 с 2013 года. Исторические данные — на листах
`2.1`–`2.3`, месяцы 2026 года — на листе `8.1` в строке РФ внутри каждой отрасли.
Старые данные ОКВЭД-2007 не присоединяются.

In [6]:
name = FILES['ppi']
current = read_excel(name, '8.1')
expect(current, 0, 0, '2026', contains=True)
expect(current, 1, 0, 'к предыдущему месяцу', contains=True)
for sheet, slug, sector, code_value in [
    ('2.1', 'industry', 'Промышленность', '1323500.029.31'),
    ('2.2', 'mining', 'Добыча полезных ископаемых', 'B'),
    ('2.3', 'manufacturing', 'Обрабатывающие производства', 'C'),
]:
    frame = read_excel(name, sheet)
    expect(frame, 0, 1, 'ОКВЭД2', contains=True)
    expect(frame, 0, 1, sector, contains=True)
    expect(frame, 3, 0, 'К предыдущему месяцу')
    records = []
    for column, year in enumerate(range(2013, 2026), 1):
        if year_number(frame.iat[2, column]) != year:
            raise ValueError(f'Изменился год цен производителей: {sheet}, {column}')
        for month in range(1, 13):
            row = month + 3
            expect(frame, row, 0, MONTHS[month - 1])
            records.append((pd.Timestamp(year=year, month=month, day=1), number(frame.iat[row, column])))
    add_series(records, f'ppi_{slug}_mom_index', 'monthly', '%', 'предыдущий месяц = 100',
               name, sheet, 'ОКВЭД2; товары для реализации на внутреннем рынке.')
    records = []
    matches = current.index[current[1].map(text).eq(code_value)].tolist()
    if len(matches) != 1:
        raise ValueError(f'Не найдена однозначная отрасль: {sector}')
    row = matches[0] + 1
    expect(current, row, 0, 'Российская Федерация')
    for month in range(1, 10):
        expect(current, 2, month + 1, MONTHS[month - 1])
        records.append((pd.Timestamp(year=2026, month=month, day=1), number(current.iat[row, month + 1])))
    add_series(records, f'ppi_{slug}_mom_index', 'monthly', '%', 'предыдущий месяц = 100',
               name, '8.1', 'ОКВЭД2; товары для реализации на внутреннем рынке.')
print('Цены производителей: извлечены 3 ряда.')

Цены производителей: извлечены 3 ряда.


## 7. Номинальная зарплата
Берём месяцы с 1998 года: более ранние значения выражены в тысячах рублей.
Средние за год и квартал исключены. В трёх ячейках конца 2022 года отдельно
обработана сноска `(2)`; её территориальное пояснение сохранено.

In [7]:
name, sheet = FILES['wages'], 'Лист1'
frame = read_excel(name, sheet)
expect(frame, 0, 0, 'номинальная начисленная заработная плата', contains=True)
expect(frame, 1, 0, 'в целом по экономике Российской Федерации', contains=True)
expect(frame, 3, 0, 'Рублей', contains=True)
footnoted = {(38, 15): '62 470(2)', (38, 16): '63 060(2)', (38, 17): '88 468(2)'}
records = []
for row, expected_year in zip(range(14, 43), range(1998, 2027)):
    year = year_number(frame.iat[row, 0])
    if year != expected_year:
        raise ValueError(f'Изменился год зарплаты: строка {row}')
    for month in range(1, 13):
        column = month + 5
        expect(frame, 4, column, MONTHS[month - 1])
        value = frame.iat[row, column]
        if (row, column) in footnoted:
            if text(value) != footnoted[row, column]:
                raise ValueError('Изменилась проверенная ячейка зарплаты со сноской')
            value = text(value).removesuffix('(2)')
        records.append((pd.Timestamp(year=year, month=month, day=1), number(value)))
add_series(records, 'nominal_wages_rub', 'monthly', 'рублей',
           'номинальная начисленная зарплата, рублей в месяц', name, sheet,
           'Ряд с 1998 года; территориальные изменения см. source_notes.csv.')
print('Зарплата: извлечён 1 ряд.')

Зарплата: извлечён 1 ряд.


## 8. Розничная торговля
Текущий файл содержит только номинальный оборот за месяцы 2026 года.
Строки «январь–март» и другие накопительные итоги исключаются.
Годовой файл `IFO_rozn_2025.xls` не используется как месячный показатель.

In [8]:
name, sheet = FILES['retail'], '1'
frame = read_excel(name, sheet)
expect(frame, 2, 0, 'Оборот розничной торговли по месяцам по Российской Федерации, млн.руб.')
expect(frame, 4, 1, 'всего')
if year_number(frame.iat[6, 0]) != 2026:
    raise ValueError('Изменился год оборота розницы')
records = []
for row in range(8, 30):
    label = text(frame.iat[row, 0]).lower()
    if label in MONTHS:
        month = MONTHS.index(label) + 1
        records.append((pd.Timestamp(year=2026, month=month, day=1), number(frame.iat[row, 1])))
add_series(records, 'retail_turnover_nominal_mln_rub', 'monthly', 'млн рублей',
           'оборот за отдельный месяц в текущих ценах', name, sheet,
           'Текущие цены; это не индекс физического объёма.')
print('Розница: извлечён 1 ряд.')

Розница: извлечён 1 ряд.


## 9. Физический объём платных услуг
Собираем отдельные месячные выпуски с января 2022 по сентябрь 2026.
В старых книгах строка России находится на одну строку ниже из-за дополнительной
нумерации столбцов; проверяются оба известных формата и база каждого индекса.
Накопительный индекс к соответствующему периоду прошлого года исключается.

In [9]:
sheet = 'платные всего'
for year, month, name in SERVICES:
    frame = read_excel(name, sheet)
    expect(frame, 1, 0, 'Основные показатели платных услуг населению', contains=True)
    expect(frame, 1, 0, f'{MONTHS[month - 1]} {year}', contains=True)
    expect(frame, 2, 0, 'Платные услуги населению')
    expect(frame, 4, 2, 'Индекс физического объема', contains=True)
    rows = [r for r in (6, 7) if re.fullmatch(
        r'(?:Россия|Российская Федерация)(?:\*+|\d+\)|\(\d+\))?', text(frame.iat[r, 0]))]
    if len(rows) != 1:
        raise ValueError(f'Не найдена однозначная строка России: {name}')
    for column, slug, header, basis in [
        (2, 'mom', 'предыдущему месяцу', 'предыдущий месяц = 100'),
        (3, 'yoy', 'соответствующему месяцу прошлого года', 'соответствующий месяц предыдущего года = 100'),
    ]:
        expect(frame, 5, column, header)
        add_series([(pd.Timestamp(year=year, month=month, day=1), number(frame.iat[rows[0], column]))],
                   f'paid_services_volume_{slug}_index', 'monthly', '%', basis, name, sheet,
                   'Отдельные выпуски; история может отличаться от позднейших пересмотренных рядов.')
print(f'Услуги: 2 ряда из {len(SERVICES)} месячных выпусков.')

Услуги: 2 ряда из 57 месячных выпусков.


## 10. Квартальный ВВП
Используем современные листы с единой методологией. Постоянные цены 2021 года
сохраняются отдельно от индекса к соответствующему кварталу предыдущего года.
Кварталы не интерполируются в месяцы.

In [10]:
name = FILES['gdp']
quarters = ['I квартал', 'II квартал', 'III квартал', 'IV квартал']
for sheet, indicator, unit, basis, start_year in [
    ('9', 'gdp_real_2021_bln_rub', 'млрд рублей', 'постоянные цены 2021 года', 2011),
    ('10', 'gdp_real_2021_sa_bln_rub', 'млрд рублей', 'постоянные цены 2021 года, сезонная корректировка', 2011),
    ('12', 'gdp_volume_yoy_index', '%', 'соответствующий квартал предыдущего года = 100', 2012),
]:
    frame = read_excel(name, sheet)
    if sheet == '12':
        expect(frame, 1, 0, 'Индексы физического объема валового внутреннего продукта', contains=True)
        expect(frame, 1, 0, 'к соответствующему кварталу предыдущего года', contains=True)
    else:
        expect(frame, 1, 0, 'в ценах 2021г., млрд.руб.', contains=True)
        if ('с исключением сезонного фактора' in text(frame.iat[1, 0])) != (sheet == '10'):
            raise ValueError('Изменился вариант сезонной корректировки ВВП')
    records = []
    for column in range(frame.shape[1]):
        year, quarter = start_year + column // 4, column % 4 + 1
        if year_number(frame.iat[2, column - column % 4]) != year:
            raise ValueError(f'Изменился год ВВП: {sheet}, {column}')
        expect(frame, 3, column, quarters[quarter - 1])
        records.append((pd.Timestamp(year=year, month=quarter * 3 - 2, day=1), number(frame.iat[4, column])))
    add_series(records, indicator, 'quarterly', unit, basis, name, sheet,
               'Современная методология; старые листы с другими базовыми ценами не присоединяются.')
print('ВВП: извлечены 3 квартальных ряда.')

ВВП: извлечены 3 квартальных ряда.


## 11. Проверки и сохранение CSV
Проверяем дубли, пропуски и единицы; дополнительно сверяем четыре ряда ИПЦ
с независимо расположенными строками РФ в региональной книге.
Пустые будущие месяцы/кварталы отсекаются, внутренние пропуски сохраняются.

Результат: `macro_long.csv`, `macro_monthly.csv`, `macro_quarterly.csv`,
`quality_report.csv`, `source_notes.csv`. Время получения файла не заменяет дату
публикации исторического наблюдения: `release_date` пока пустая. Для бэктеста
её и историю пересмотров нужно восстановить отдельно.

In [11]:
macro = pd.concat(parts, ignore_index=True).sort_values(['indicator', 'period']).reset_index(drop=True)
if macro.duplicated(['indicator', 'period']).any():
    raise ValueError('Повторяются периоды одного показателя')
for indicator, group in macro.groupby('indicator'):
    if any(group[column].nunique() != 1 for column in ['frequency', 'unit', 'basis', 'territory']):
        raise ValueError(f'Несовместимые измерения: {indicator}')

values = macro.set_index(['indicator', 'period']).value
checked = 0
with pd.ExcelFile(paths[FILES['cpi_check']]) as book:
    for sheet in book.sheet_names:
        match = re.fullmatch(r'(\d{2})\((\d{4})\)', sheet)
        if match is None:
            continue
        month, year = map(int, match.groups())
        frame = read_excel(FILES['cpi_check'], sheet)
        expect(frame, 4, 1, 'Российская Федерация')
        expect(frame, 1, 2, 'к предыдущему месяцу', contains=True)
        for column, indicator in enumerate(CPI_INDICATORS, 2):
            actual = number(frame.iat[4, column])
            expected = values.loc[(indicator, pd.Timestamp(year=year, month=month, day=1))]
            if not math.isclose(actual, expected, rel_tol=0, abs_tol=1e-8):
                raise ValueError(f'Не совпал ИПЦ: {sheet}, {indicator}')
            checked += 1

quality_rows = []
matrices = {}
for indicator, group in macro.groupby('indicator'):
    rule = {'monthly': 'MS', 'quarterly': 'QS'}[group.frequency.iloc[0]]
    calendar = pd.date_range(group.period.min(), group.period.max(), freq=rule)
    series = group.set_index('period').value.reindex(calendar)
    quality_rows.append(dict(indicator=indicator, frequency=group.frequency.iloc[0],
                             start=calendar.min(), end=calendar.max(),
                             observed=int(series.notna().sum()), missing=int(series.isna().sum())))
quality = pd.DataFrame(quality_rows)
for frequency, group in macro.groupby('frequency'):
    wide = group.pivot(index='period', columns='indicator', values='value').sort_index()
    calendar = pd.date_range(wide.index.min(), wide.index.max(), freq={'monthly': 'MS', 'quarterly': 'QS'}[frequency])
    matrices[frequency] = wide.reindex(calendar).rename_axis('period')

OUT.mkdir(parents=True, exist_ok=True)
macro.to_csv(OUT / 'macro_long.csv', index=False, date_format='%Y-%m-%d')
for frequency, wide in matrices.items():
    wide.to_csv(OUT / f'macro_{frequency}.csv', date_format='%Y-%m-%d')
quality.to_csv(OUT / 'quality_report.csv', index=False, date_format='%Y-%m-%d')
pd.DataFrame(sorted(source_notes), columns=['file_name', 'sheet', 'note']).to_csv(OUT / 'source_notes.csv', index=False)
display(quality)
display(macro[['indicator', 'period', 'value', 'frequency', 'unit']].head())
print(f'Показателей: {macro.indicator.nunique()}; значений: {macro.value.notna().sum()}')
print(f'Независимая сверка ИПЦ: {checked} значений совпали')
print('Результаты:', OUT)

,indicator,frequency,start,end,observed,missing
0,cpi_food_mom_index,monthly,1991-01-01,2026-09-01,429,0
1,cpi_mom_index,monthly,1991-01-01,2026-09-01,429,0
2,cpi_nonfood_mom_index,monthly,1991-01-01,2026-09-01,429,0
3,cpi_services_mom_index,monthly,1991-01-01,2026-09-01,429,0
4,gdp_real_2021_bln_rub,quarterly,2011-01-01,2026-04-01,62,0
5,gdp_real_2021_sa_bln_rub,quarterly,2011-01-01,2026-04-01,62,0
6,gdp_volume_yoy_index,quarterly,2012-01-01,2026-04-01,58,0
7,nominal_wages_rub,monthly,1998-01-01,2026-08-01,344,0
8,paid_services_volume_mom_index,monthly,2022-01-01,2026-09-01,57,0
9,paid_services_volume_yoy_index,monthly,2022-01-01,2026-09-01,57,0


,indicator,period,value,frequency,unit
0,cpi_food_mom_index,1991-01-01,104.6,monthly,%
1,cpi_food_mom_index,1991-02-01,103.1,monthly,%
2,cpi_food_mom_index,1991-03-01,105.2,monthly,%
3,cpi_food_mom_index,1991-04-01,171.8,monthly,%
4,cpi_food_mom_index,1991-05-01,100.5,monthly,%


Показателей: 14; значений: 2860
Независимая сверка ИПЦ: 228 значений совпали
Результаты: /Users/oleynik_n_a/Documents/ChatGPT/Годовой проект/data/processed/20261029T214426724279Z
